# Corporacion Favorita - New Superb Forecasting Model - 

## Split and Model Pipeline

#codi

Made by 4B Consultancy (Janne Heuvelmans, Georgi Duev, Alexander Engelage, Sebastiaan de Bruin) - 2024

In this data pipeline, 

The following steps are made within this notebook:  

>-0. Import Packages 

>-1. Load final dataset and aggregate dataset to weekly level
    -1.1 Load final dataset made in Data Preperation Pipeline Notebook
    -1.2 Aggregate dataset to weekly level

>-2. Column transformers and Train, Test, Validation Split

>-3. Models

>-4. Pick best model one and optimize with grid search

## 0. Import Packages

In [1]:
# Importing the libraries
import pandas as pd
import numpy as np
import polars as pl
import os
import sys
import altair as alt
import vegafusion as vf
import sklearn
import time
from datetime import date, datetime, timedelta
from sklearn.pipeline import Pipeline, make_pipeline

In [2]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

from sklearn.metrics import mean_absolute_percentage_error

import statsmodels.api as sm

In [3]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.seasonal import seasonal_decompose

In [4]:
from sktime.forecasting.compose import EnsembleForecaster

## 1. Load final dataset

### 1.1. Functions - Import raw data from local PATH
Create import data function and give basic information function within the importing function.

Return basic information on each dataframe:  
- a) Information on the number of observation and features.  
- b) Information on the size of the dataframe. 

TO-DO: Import via polars, and use polars dataframe?

In [5]:
def f_get_data_and_info(import_path, file_name):

    print(f"\nReading file {file_name}\n")

    # Load data.
    df = pd.read_parquet(import_path + file_name + ".parquet")

    # Getting the basic information of the dataframe (number of observations and features, and size)
    print(
        f"The '{file_name}' dataframe contains: {df.shape[0]:,}".replace(",", ".")
        + f" observations and {df.shape[1]} features."
    )
    print(
        f"Prepared and transformed dataframe has optimized size of {round(sys.getsizeof(df)/1024/1024/1024, 2)} GB."
    )

    return df

### 1.2. Importing raw data
Importing parquet files with importing function (giving basic information)

In [ ]:
import_path = "C:/Users/alexander/Documents/0. Data Science and AI for Experts/FILE/"

# Importing final df
df_final = f_get_data_and_info(import_path, file_name="Prepped_data_20241211")

# df_final = f_get_data_and_info(import_path, file_name="df_test_with_forecasts_20241120")

## 2.0 Train Test Val Split

train_test_val_split without creating X (features) and y (target)

In [8]:
def train_test_val_split(df, window_length=26):

    # Sort the DataFrame by store number, item number, and date for ordering
    df = df.sort_values(["store_nbr", "item_nbr", "week_number_cum"])

    # Get the maximum week in the dataset
    max_week = df["week_number_cum"].max()

    # Calculate start and end weeks for test and validation  sets
    val_week_end = max_week - 1

    val_week_start = max_week - window_length

    test_week_start = max_week - 2 * window_length

    test_week_end = val_week_start - 1

    train_week_start = max_week - 6 * window_length

    train_week_end = test_week_start - 1

    # Train data: All data before the start of the validation period
    train = df[
        (df["week_number_cum"] >= train_week_start)
        & (df["week_number_cum"] <= train_week_end)
    ]

    # Val data: From val_week_start to val_week_end
    test = df[
        (df["week_number_cum"] >= test_week_start)
        & (df["week_number_cum"] <= test_week_end)
    ]

    # Test data: From test_week_start to max_week
    val = df[
        (df["week_number_cum"] >= val_week_start)
        & (df["week_number_cum"] <= val_week_end)
    ]

    # Function to print split information
    def print_split_info(split_name, split):
        print(f"\n{split_name} set: shape: {split.shape}")
        print(f"{split_name} Min Week: {split['week_number_cum'].min()}")
        print(f"{split_name} Min Date: {split['date'].min()}")
        print(f"{split_name} Max Week: {split['week_number_cum'].max()}")
        print(f"{split_name} Max Date: {split['date'].max()}")
        print(f"{split_name} number of weeks: {split['week_number_cum'].nunique()}")
        print(f"Number of stores: {split['store_nbr'].nunique()}")
        print(f"Number of items: {split['item_nbr'].nunique()}")
        print(f"Size of {round(sys.getsizeof(split)/1024/1024/1024, 2)} GB.")

    # Print information about the splits
    print_split_info("Train", train)
    print_split_info("Test", test)
    print_split_info("Validation", val)

    return train, test, val

In [9]:
train, test, val = train_test_val_split(df_final, window_length=26)

## 3.0 Functions - Impute stockouts and Aggregate dataset to weekly level


#### 3.1. Impute stockouts

Stockout on store level

•      Perishable good: when there are missing values for two consecutive days for a given item per individual store 

•      Nonperishable goods: when there are missing values for 7 consecutive days for a given item and per individual store

•      Action: Impute with Rolling Mean with defeault window of 7 days 

------------------------------------

In [10]:
def impute_stockouts_polars(df_pandas, window_size=7):
    # Convert the input Pandas DataFrame to a Polars DataFrame
    df = pl.from_pandas(df_pandas)

    # Sort the DataFrame by store number, item number, and date for proper ordering
    df = df.sort(["store_nbr", "item_nbr", "date"])

    # Create a boolean column to indicate where 'unit_sales' is missing
    df = df.with_columns((pl.col("unit_sales").is_null()).alias("is_missing"))

    # Assign a group identifier to each segment of missing or non-missing values
    df = df.with_columns(
        (
            pl.col("is_missing").cast(pl.Int16)
            != pl.col("is_missing").cast(pl.Int16).shift(1)
        )
        .cast(pl.Int16)
        .cum_sum()
        .alias("missing_group")
        .cast(pl.Int32)
    )

    # Calculate cumulative count of missing values within each segment of missing data
    df = df.with_columns(
        pl.when(pl.col("is_missing"))
        .then(
            pl.col("is_missing")
            .cast(pl.Int16)
            .cum_sum()
            .over(["store_nbr", "item_nbr", "missing_group"])
        )
        .otherwise(0)
        .alias("missing_count")
    )

    # Identify groups to find maximum of the same missing_count group
    df = df.with_columns(
        pl.col("missing_count")
        .max()
        .over(["missing_group"])
        .alias("group_max_missing_count")
        .cast(pl.Int16)
    )

    # Function for rolling mean imputation
    def rolling_mean_imputation(df, window_size=7):
        # Add rolling mean column for grouped data
        df = df.with_columns(
            pl.col("unit_sales")
            .rolling_mean(window_size=window_size, min_periods=1)
            .over(["store_nbr", "item_nbr"])
            .shift(1)  # Shift to exclude current row
            .alias("unit_sales_rolling_mean")
        )

        # Replace nulls in the target column with the calculated rolling mean
        df = df.with_columns(
            pl.when(pl.col("unit_sales").is_null())
            .then(pl.col("unit_sales_rolling_mean"))
            .otherwise(pl.col("unit_sales"))
            .alias("unit_sales")
        )

        # Drop the temporary rolling mean column
        df = df.drop("unit_sales_rolling_mean")

        return df

    # Apply rolling mean imputation based on perishable status
    df = df.with_columns(
        [
            pl.when(pl.col("perishable") == 1)  # If the item is perishable
            .then(
                pl.when(pl.col("group_max_missing_count") == 1)  # 1 missing value
                .then(0)  # Impute with 0
                .when(
                    pl.col("group_max_missing_count") > 2
                )  # More than 2 missing values
                .then(0)  # Impute with 0
                .when(
                    pl.col("group_max_missing_count") == 2
                )  # Exactly 2 missing values
                .then(
                    rolling_mean_imputation(df, window_size=7)["unit_sales"]
                )  # Impute with rolling mean for 2 missing days
                .otherwise(pl.col("unit_sales"))  # Keep original value
            )
            .when(pl.col("perishable") == 0)  # If the item is not perishable
            .then(
                pl.when(
                    pl.col("group_max_missing_count") > 7
                )  # More than 7 missing values
                .then(0)  # Impute with 0
                .when(
                    pl.col("group_max_missing_count") <= 7
                )  # 7 or fewer missing values
                .then(
                    rolling_mean_imputation(df, window_size=7)["unit_sales"]
                )  # Impute with rolling mean for missing 7 or fewer days
                .otherwise(pl.col("unit_sales"))  # Keep original value
            )
            .otherwise(pl.col("unit_sales"))  # For other cases, keep the original value
            .alias("unit_sales")
        ]
    )

    df = df.drop(
        "is_missing", "missing_group", "missing_count", "group_max_missing_count"
    )

    # Convert Polars df back to Pandas df
    df = df.to_pandas()

    return df

### 3.2. Aggregate dataset to weekly level

- Group the DataFrame by store number, item number, year, and week_cum_number, then aggregate the columns
--> "unit_sales","onpromotion", "holiday_local_count","holiday_regional_count","holiday_national_count",


In [11]:
def aggregate_week(df):

    # Sort the DataFrame by store number, item number, and date for ordering
    df = df.sort_values(["store_nbr", "item_nbr", "year", "week_nbr"])

    # Group by the specified columns and aggregate
    df = (
        df.groupby(
            [
                "store_nbr",
                "item_nbr",
                "year",
                "week_number_cum",  # Aggregating by week_number_cum
            ]
        )
        .agg(
            {
                "unit_sales": "sum",
                "onpromotion": "sum",
                "holiday_local_count": "sum",
                "holiday_regional_count": "sum",
                "holiday_national_count": "sum",
                "date": "first",  # Keep the first day of week, needed to run Timeseries models from SKtime
                "store_type": "first",  # Keep the first occurrence of store_type
                "store_cluster": "first",  # Keep the first occurrence of store_cluster
                "item_family": "first",  # Keep the first occurrence of item_family
                "item_class": "first",  # Keep the first occurrence of item_class
                "perishable": "first",  # Keep the first occurrence of perishable
                "store_status": "last",  # Keep the last occurrence of store_status
                "item_status": "last",  # Keep the last occurrence of item_status
            }
        )
        .reset_index()
    )

    return df

## 4. Pipeline and preprocessing

Splitting and preprocessing with imputation and aggregating to weekly data

In [12]:
# features = [
#     "date",
#     "store_nbr",
#     "item_nbr",  # , 'item_family', 'store_type', 'perishable'
# ]


# target_variable = ["unit_sales"]

In [13]:
features = [
    "store_nbr",
    "item_nbr",
    "date",
    "onpromotion",
    # "holiday_local_count",
    # "holiday_national_count",
    # "holiday_regional_count",
    "store_type",
    "store_cluster",
    "item_family",
    "item_class",
    "perishable",
    # "store_status",
    # "item_status",
    "year",
    "week_number_cum",
]

target_variable = ["unit_sales"]

In [14]:
def impute_agg_preprocessing(df, window_size=7):

    df = impute_stockouts_polars(df, window_size)

    df = aggregate_week(df)

    return df

In [15]:
def preprocess_split_filter(df, features, target_variable):

    # Splitting in train, validation, test split
    print(f"\nStep 1: Splitting in train, validation, test split")
    train_df, test_df, val_df = train_test_val_split(df)

    # Preprocessing with imputation and aggregating to weekly data
    print(f"\nStep 2: Preprocessing with imputation and aggregating to weekly data")
    train_df = impute_agg_preprocessing(train_df)
    test_df = impute_agg_preprocessing(test_df)
    val_df = impute_agg_preprocessing(val_df)

    # Filter spilts on needed feature and target variables
    print(f"\nStep 3: Filter spilts on needed feature and target variables")
    train_df = train_df[features + target_variable]
    test_df = test_df[features + target_variable]
    val_df = val_df[features + target_variable]

    # Ensure df's are sorted by store, item, and date for alignment
    print(f"\nStep 4: Ensure dfs are sorted by store, item, and date for alignment")
    train_df = train_df.sort_values(by=["store_nbr", "item_nbr", "date"])
    test_df = test_df.sort_values(by=["store_nbr", "item_nbr", "date"])
    val_df = val_df.sort_values(by=["store_nbr", "item_nbr", "date"])

    return train_df, val_df, test_df

In [ ]:
train_df, test_df, val_df = preprocess_split_filter(df_final, features, target_variable)

In [ ]:
# filter for item 103520 for store 10 to debug HW model
# df_final = df_final[(df_final["store_nbr"] == 10) & (df_final["item_nbr"] == 103520)]

### Write to Parquet fil and saves it in output_path

In [18]:
def save_dataframe_to_parquet(df, output_path, file_prefix="Prepped_data"):
    try:
        # Ensure the directory exists
        os.makedirs(output_path, exist_ok=True)

        # Generate today's date for the filename
        today = date.today().strftime("%Y%m%d")

        # Create the full filename with path
        filename = f"{file_prefix}_{today}.parquet"
        full_path = os.path.join(output_path, filename)

        # Save the DataFrame to a Parquet file
        df.to_parquet(full_path)

        print(f"DataFrame successfully saved to {full_path}")

        return full_path

    except Exception as e:
        print(f"Error saving DataFrame to Parquet file: {e}")

        return None

In [19]:
output_path = "C:/Users/alexander/Documents/0. Data Science and AI for Experts/FILE"

In [20]:
# train_df_saved_path = save_dataframe_to_parquet(
#    train_df, output_path, file_prefix="train_df"
# )

In [ ]:
def stop

## 5. Model Pipeline

### 5.1 Model: Holt-Winters

In [ ]:
# holt_winters_model = ExponentialSmoothing(
#     df_train, trend="add", seasonal="add", seasonal_periods=12
# ).fit()

#                   .fit(smoothing_level=0.5, #=best_alpha
#                        smoothing_slope=0.5, #=best_beta
#                        smoothing_seasonal=0.5) #=best_gamma

### 5.2 Run Model

test

In [ ]:
def perform_seasonal_decomposition(series, seasonal_periods):

    try:
        # Use multiplicative decomposition if data is always positive
        if (series > 0).all():
            decomposition = seasonal_decompose(
                series, model="multiplicative", period=seasonal_periods
            )

        else:
            decomposition = seasonal_decompose(
                series, model="additive", period=seasonal_periods
            )
        return decomposition

    except Exception as e:

        print(f"Decomposition failed: {e}")

        return None

In [ ]:
def preprocess_series(series, decomposition=None):

    # Remove extreme outliers
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # Clip values to remove extreme outliers
    preprocessed_series = series.clip(lower=lower_bound, upper=upper_bound)

    # If decomposition is available, you could use residuals for further processing
    if decomposition is not None:
        # Example: Remove strong residual effects
        preprocessed_series = preprocessed_series - decomposition.resid

    return preprocessed_series

In [35]:
def extract_model_parameters(fitted_model, store, item):

    params = {
        "store_nbr": store,
        "item_nbr": item,
        "initial_alpha": fitted_model.params.get("initial_level", None),
        "initial_beta": fitted_model.params.get("initial_trend", None),
        "initial_gamma": fitted_model.params.get("initial_seasonal", None),
        "damping": fitted_model.params.get("damping_slope", None),
        "alpha": fitted_model.model.params.get("smoothing_level", None),
        "beta": fitted_model.model.params.get("smoothing_slope", None),
        "gamma": fitted_model.model.params.get("smoothing_seasonal", None),
        # Additional derived metrics
        # "aic": fitted_model.aic,
        # "bic": fitted_model.bic
    }
    return params

In [ ]:
def visualize_decomposition(decomposition, store, item):

    if decomposition is not None:
        plt.figure(figsize=(12, 8))
        plt.suptitle(f"Seasonal Decomposition - Store {store}, Item {item}")

        plt.subplot(411)
        plt.title("Original")
        plt.plot(decomposition.observed)

        plt.subplot(412)
        plt.title("Trend")
        plt.plot(decomposition.trend)

        plt.subplot(413)
        plt.title("Seasonal")
        plt.plot(decomposition.seasonal)

        plt.subplot(414)
        plt.title("Residual")
        plt.plot(decomposition.resid)

        plt.tight_layout()
        # Instead of plt.show(), you might want to save or log the plot
        plt.savefig(f"decomposition_store_{store}_item_{item}.png")
        plt.close()

In [ ]:
def holt_winters_model_forecast(
    train_df, forecast_df, seasonal_periods=52, trend="add", seasonal="add"
):
    # Unique store and item combinations
    unique_stores = train_df["store_nbr"].unique()
    unique_items = train_df["item_nbr"].unique()

    forecasts = {}
    fitted_values = {}
    model_params = []
    decomposition_results = {}

    for store in unique_stores:
        print(f"Processing store {store}")

        for item in unique_items:

            # Filter the data for the specific store and item
            train_data = train_df[
                (train_df["store_nbr"] == store) & (train_df["item_nbr"] == item)
            ].copy()

            # Convert date to datetime and set as index with Weekly frequency, starting on Monday
            train_data["date"] = pd.to_datetime(train_data["date"])
            train_data = train_data.set_index("date")
            train_data = train_data.asfreq("W-MON")

            # Extract the unit_sales series
            train_series = train_data["unit_sales"].copy()

            # Perform Seasonal Decomposition
            try:
                # Handle potential issues with decomposition
                if len(train_series) < seasonal_periods * 2:
                    print(
                        f"Warning: Not enough data for meaningful decomposition for store {store}, item {item}"
                    )
                    decomposition = None

                else:

                    decomposition = perform_seasonal_decomposition(
                        train_series, seasonal_periods
                    )

                    # Visualization of decomposition (can be commented out or logged)
                    visualize_decomposition(decomposition, store, item)

                # Preprocess series based on decomposition insights
                preprocessed_series = preprocess_series(train_series, decomposition)

                # Fit Holt-Winters Model
                model = ExponentialSmoothing(
                    preprocessed_series,
                    trend=trend,
                    seasonal=seasonal,
                    seasonal_periods=seasonal_periods,
                )

                # Advanced model fitting with parameter optimization
                fitted_model = model.fit(optimized=True, use_brute=True)

                # Forecast
                forecast_length = len(
                    forecast_df[
                        (forecast_df["store_nbr"] == store)
                        & (forecast_df["item_nbr"] == item)
                    ]["date"].unique()
                )

                forecast = fitted_model.forecast(forecast_length)

                # Store results
                forecasts[(store, item)] = forecast
                fitted_values[(store, item)] = fitted_model.fittedvalues

                # Store decomposition if available
                if decomposition is not None:
                    decomposition_results[(store, item)] = {
                        "trend": decomposition.trend,
                        "seasonal": decomposition.seasonal,
                        "resid": decomposition.resid,
                    }

                # Extract model parameters
                params = extract_model_parameters(fitted_model, store, item)
                model_params.append(params)

            except Exception as e:
                print(f"Model failed for store {store}, item {item}: {e}")
                continue

    # Convert model parameters to dataframe
    params_df = pd.DataFrame(model_params)

    return forecasts, fitted_values, params_df, decomposition_results

In [ ]:
# forecasts, fitted_values, params_df, decomposition_results = (
#     holt_winters_model_forecast(
#         train_df, test_df, seasonal_periods=52, trend="add", seasonal="add"
#     )
# )

# # decomposition results
# for (store, item), decomp in decomposition_results.items():
#     print(f"Store {store}, Item {item} Decomposition:")
#     print(decomp.keys())  # Access trend, seasonal, resid

org

In [25]:
def holt_winters_model_forecast(
    train_df, forecast_df, seasonal_periods=52, trend="add", seasonal="add"
):
    # Unique store and item combinations
    unique_stores = train_df["store_nbr"].unique()
    unique_items = train_df["item_nbr"].unique()

    forecasts = {}
    fitted_values = {}
    model_params = []

    for store in unique_stores:
        print(f"Model Hollt-Winters starts training for store {store}")

        for item in unique_items:
            # Filter the data for the specific store and item
            train_data = train_df[
                (train_df["store_nbr"] == store) & (train_df["item_nbr"] == item)
            ].copy()

            # Convert date to datetime and set as index
            train_data["date"] = pd.to_datetime(train_data["date"])
            train_data = train_data.set_index("date")
            train_data = train_data.asfreq(
                "W-MON"
            )  # Setting frequency to weekly, starting week at Monday

            # Extract the unit_sales series
            train_series = train_data["unit_sales"].copy()

            # Fit Holt-Winters Model
            try:
                model = ExponentialSmoothing(
                    train_series,
                    trend=trend,
                    seasonal=seasonal,
                    seasonal_periods=seasonal_periods,
                )

                fitted_model = model.fit()
                # fitted_model = model.fit(optimized=True, use_brute=True)

                # Forecast the forecast period length
                forecast_length = len(
                    forecast_df[
                        (forecast_df["store_nbr"] == store)
                        & (forecast_df["item_nbr"] == item)
                    ]["date"].unique()
                )

                forecast = fitted_model.forecast(forecast_length)

                # Store the forecast
                forecasts[(store, item)] = forecast
                fitted_values[(store, item)] = fitted_model.fittedvalues

                # Extract model parameters
                params = {
                    "store_nbr": store,
                    "item_nbr": item,
                    "initial_alpha": fitted_model.params.get("initial_level", None),
                    "initial_beta": fitted_model.params.get("initial_trend", None),
                    "initial_gamma": fitted_model.params.get("initial_seasonal", None),
                    "damping": fitted_model.params.get("damping_slope", None),
                    "alpha": fitted_model.model.params.get("smoothing_level", None),
                    "beta": fitted_model.model.params.get("smoothing_slope", None),
                    "gamma": fitted_model.model.params.get("smoothing_seasonal", None),
                }
                model_params.append(params)

            except Exception as e:
                print(f"Model failed for store {store}, item {item}: {e}")

    # Convert model parameters to df
    params_df = pd.DataFrame(model_params)

    return forecasts, fitted_values, params_df

In [ ]:
from pandas.plotting import autocorrelation_plot

autocorrelation_plot(train_df["unit_sales"])

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose
import matplotlib.pyplot as plt

result = seasonal_decompose(train_df["unit_sales"], period=52)
result.plot()
plt.show()

In [ ]:
DEF STOP

In [ ]:
# train_val_df = pd.concat([train_df, val_df], ignore_index=True)

In [ ]:
# train_df_store_1 = train_df[train_df["store_nbr"] == 1]
# val_df_store_1 = val_df[val_df["store_nbr"] == 1]
# test_df_store_1 = test_df[test_df["store_nbr"] == 1]

# train_df_store_10_103520 = train_df_store_1[train_df_store_1["item_nbr"] == 103520]
# val_df_store_10_103520 = val_df_store_1[val_df_store_1["item_nbr"] == 103520]
# test_df_store_10_103520 = test_df_store_1[test_df_store_1["item_nbr"] == 103520]

In [ ]:
# train_df, test_df, val_df

In [ ]:
test_forecasts, test_fitted_values, train_params_df = holt_winters_model_forecast(
    train_df,
    test_df,
    seasonal_periods=52,
    trend="add",
    seasonal="add",
)

In [ ]:
train_params_df_save = save_dataframe_to_parquet(
    train_params_df, output_path, file_prefix="train_params_df"
)

In [ ]:
train_params_df

In [ ]:
params_df

### 5.3. Evaulation Metrics and Evaluate Model functions

In [30]:
def calculate_metrics(y_true, y_pred):

    mape = mean_absolute_percentage_error(y_true, y_pred)

    accuracy = 1 - mape

    bias = np.mean(y_pred - y_true)

    return {"MAPE": mape, "Accuracy": accuracy, "Bias": bias}

In [31]:
def evaluate_forecasts(forecasts, forcast_df):

    # Initialize lists to store metrics for each store and item
    all_metrics = []
    store_metrics = {}

    # Iterate over each store and item combination to collect true and predicted values
    for (store, item), forecast in forecasts.items():

        y_true = forcast_df[
            (forcast_df["store_nbr"] == store) & (forcast_df["item_nbr"] == item)
        ]["unit_sales"]
        y_pred = forecast

        # Index the values
        # y_true = y_true.reset_index(drop=True)
        # y_pred = pd.Series(forecast).reset_index(drop=True)

        if len(y_true) == 0 or len(y_pred) == 0:
            print(f"Skipping store {store}, item {item} due to empty data.")
            continue

        metrics = calculate_metrics(y_true, y_pred)
        metrics.update({"store_nbr": store, "item_nbr": item})
        all_metrics.append(metrics)

        # Initialize metrics for the store if not already present
        if store not in store_metrics:
            store_metrics[store] = {"MAPE": [], "Accuracy": [], "Bias": []}

        store_metrics[store]["MAPE"].append(metrics["MAPE"])
        store_metrics[store]["Accuracy"].append(metrics["Accuracy"])
        store_metrics[store]["Bias"].append(metrics["Bias"])

    # Calculate average metrics for each store
    average_store_metrics = []
    for store, metrics in store_metrics.items():
        average_mape = np.nanmean(metrics["MAPE"])
        average_accuracy = np.nanmean(metrics["Accuracy"])
        average_bias = np.nanmean(metrics["Bias"])

        average_store_metrics.append(
            {
                "store_nbr": store,
                "item_nbr": "average",
                "MAPE": average_mape,
                "Accuracy": average_accuracy,
                "Bias": average_bias,
            }
        )

    # Calculate overall average metrics
    overall_mape = np.nanmean(
        [metric["MAPE"] for metric in all_metrics if not np.isnan(metric["MAPE"])]
    )
    overall_accuracy = np.nanmean(
        [
            metric["Accuracy"]
            for metric in all_metrics
            if not np.isnan(metric["Accuracy"])
        ]
    )
    overall_bias = np.nanmean(
        [metric["Bias"] for metric in all_metrics if not np.isnan(metric["Bias"])]
    )

    overall_metrics = {
        "store_nbr": "overall",
        "item_nbr": "overall",
        "MAPE": overall_mape,
        "Accuracy": overall_accuracy,
        "Bias": overall_bias,
    }

    # Convert metrics to df's
    metrics_df = pd.DataFrame(all_metrics)
    average_store_metrics_df = pd.DataFrame(average_store_metrics)
    overall_metrics_df = pd.DataFrame([overall_metrics])

    # Print metrics
    # print(metrics_df)
    # print(average_store_metrics_df)
    print(overall_metrics_df)

    return metrics_df, average_store_metrics_df, overall_metrics_df

In [ ]:
# Evaluate forecasts and print metrics
test_metrics_df, test_average_store_metrics_df, test_df_overall_metrics_df = (
    evaluate_forecasts(test_forecasts, test_df)
)

In [ ]:
# Evaluate forecasts and print metrics
test_metrics_df, test_average_store_metrics_df, test_df_overall_metrics_df = (
    evaluate_forecasts(forecasts, test_df)
)

In [ ]:
val_average_store_metrics_df

In [ ]:
val_metrics_df.sort_values(by="MAPE", ascending=True).head(30)

In [ ]:
# Evaluate forecasts and print metrics
test_metrics_df, test_average_store_metrics_df, test_overall_metrics_df = (
    evaluate_forecasts(test_forecasts, test_df)
)

In [ ]:
test_average_store_metrics_df.head(30)

In [ ]:
test_metrics_df.sort_values(by="MAPE", ascending=True).head(30)

### 5.X Function to join y_pred to Y_ptrue in original df

In [ ]:
def add_forecasts_to_df(original_df, forecasts):

    # Rename unit_sales to y_true
    original_df = original_df.rename(columns={"unit_sales": "y_true"})

    # Ensure the original DataFrame has a datetime index
    original_df["date"] = pd.to_datetime(original_df["date"])
    original_df = original_df.set_index(["date", "store_nbr", "item_nbr"])

    # Prepare a DataFrame for forecasts
    forecast_entries = []

    for (store, item), forecast_series in forecasts.items():
        # Convert Forecast servies to df
        forecast_df = forecast_series.reset_index()

        # Rename columns to y_pred and add store and item columns
        forecast_df.columns = ["date", "y_pred"]
        forecast_df["store_nbr"] = store
        forecast_df["item_nbr"] = item
        forecast_entries.append(forecast_df)

    # Combine all forecast entries
    forecast_df = pd.concat(forecast_entries, ignore_index=True)
    forecast_df["date"] = pd.to_datetime(forecast_df["date"])
    forecast_df = forecast_df.set_index(["date", "store_nbr", "item_nbr"])

    # Join the forecasted values with the original df
    result_df = original_df.join(forecast_df, how="left")

    result_df = result_df.reset_index()

    # Specify the desired column order
    desired_order = [
        "date",
        "week_number_cum",
        "store_nbr",
        "item_nbr",
        "y_true",
        "y_pred",
        "perishable",
        "store_type",
        "store_cluster",
        "item_family",
        "item_class",
    ]
    # Add the rest of the columns to the desired order
    all_columns = list(result_df.columns)
    remaining_columns = [col for col in all_columns if col not in desired_order]
    final_order = desired_order + remaining_columns

    # Reorder the DataFrame columns
    result_df = result_df[final_order]

    return result_df, forecast_df

In [ ]:
# df_val_with_forecasts = add_forecasts_to_df(val_df, val_forecasts)

# df_val_with_forecasts = save_dataframe_to_parquet(
#     df_val_with_forecasts, output_path, file_prefix="df_val_with_forecasts"
# ) val_metrics_df

In [ ]:
df_test_with_forecasts, test_forecast_df = add_forecasts_to_df(test_df, test_forecasts)

df_test_with_forecasts = save_dataframe_to_parquet(
    df_test_with_forecasts, output_path, file_prefix="df_test_with_forecasts_optimized"
)

In [ ]:
df_test_with_forecasts_seperate = save_dataframe_to_parquet(
    test_forecast_df,
    output_path,
    file_prefix="df_test_with_forecasts_seperate_optimized",
)

In [ ]:
test_forecast_df.head(30)

In [ ]:
test_metrics_df = save_dataframe_to_parquet(
    test_metrics_df, output_path, file_prefix="test_metrics_df"
)

-------------------------------------------

## 6. Hyperparameter Optimization (Grid Search approach)

https://www.kaggle.com/code/mehmetisik/smoothing-methods-holt-winters/notebook#Final-TES-Model

- alpha: smoothing level for the level components
- beta: smoothing level for the trend components
- gamma: smoothing level for the seasonal components

--> Problem, bc goining to test for every store-item combination multiple alpha, beta, and gamma. So gpu Parallel Processing needed probably

In [ ]:
train_df_store_10 = train_df[train_df["store_nbr"] == 10]
val_df_store_10 = val_df[val_df["store_nbr"] == 10]
test_df_store_10 = test_df[test_df["store_nbr"] == 10]

train_df_store_10_103520 = train_df_store_10[train_df_store_10["item_nbr"] == 103520]
val_df_store_10_103520 = val_df_store_10[val_df_store_10["item_nbr"] == 103520]
test_df_store_10_103520 = test_df_store_10[test_df_store_10["item_nbr"] == 103520]

In [ ]:
import itertools
from concurrent.futures import ProcessPoolExecutor, as_completed

In [ ]:
DEF STOP RUN

In [ ]:
fitted = model.fit(optimized=True, use_brute=True)

In [ ]:
# Create a range of values for alpha, beta, and gamma
alphas = betas = gammas = np.arange(0.10, 1, 0.10)

In [ ]:
import itertools

# create all combinations of alpha, beta, and gamma
abg = list(itertools.product(alphas, betas, gammas))

In [ ]:
def hw_model_optimizer(df_train, trend="add", seasonal="add", seasonal_periods=12, abg, step=24):

    best_alpha, best_beta, best_gamma, best_mape = None, None, None, float("inf")

    for comb in abg:

        model = ExponentialSmoothing(
            df_train, trend=trend, seasonal=seasonal, seasonal_periods=seasonal_periods
        ).fit(
            smoothing_level=comb[0], smoothing_slope=comb[1], smoothing_seasonal=comb[2]
        )

        y_pred = model.forecast(step)
        mape = mean_absolute_percentage_error(df_train[-step:], y_pred)
        if mape < best_mape:
            best_alpha, best_beta, best_gamma, best_mape = (
                comb[0],
                comb[1],
                comb[2],
                mape,
            )
        print([round(comb[0], 2), round(comb[1], 2), round(comb[2], 2), round(mape, 2)])

    print(
        "best_alpha:",
        round(best_alpha, 2),
        "best_beta:",
        round(best_beta, 2),
        "best_gamma:",
        round(best_gamma, 2),
        "best_mae:",
        round(best_mape, 4),
    )

    return best_alpha, best_beta, best_gamma, best_mape

In [ ]:
best_alpha, best_beta, best_gamma, best_mae = hw_model_optimizer(df_train, abg)

In [ ]:
final_tes_model = ExponentialSmoothing(
    train, trend="add", seasonal="add", seasonal_periods=12
).fit(
    smoothing_level=best_alpha, smoothing_trend=best_beta, smoothing_seasonal=best_gamma
)

Parallel Processing for item forecasts in parallel for each unique store

In [ ]:
import itertools

t_params = ['add', 'mul', None]
d_params = [True, False]
s_params = ['add', 'mul', None]
alphas = betas = gammas = np.arange(0, 1.01, 0.01)
abg_combinations = list(itertools.product(t_params, d_params, s_params, alphas, betas, gammas)) -->

#### 6.X Paralell processing 

- BLC = fitted_model.aic
- BIC = fitted_model.bic

Optimize on BIC instead of MAPE

In [ ]:
def process_item(
    store_nbr,
    item_nbr,
    train_df,
    forecast_df,
    abg_combinations,
    seasonal_periods,
    max_no_improve_iters,
    early_stopping_threshold,
):

    try:

        # Filter the data for the specific store and item
        train_data = train_df[
            (train_df["store_nbr"] == store_nbr) & (train_df["item_nbr"] == item_nbr)
        ].copy()
        forecast_data = forecast_df[
            (forecast_df["store_nbr"] == store_nbr)
            & (forecast_df["item_nbr"] == item_nbr)
        ].copy()

        # Convert date to datetime and set as index
        # Setting frequency to weekly, start week on Monday
        train_data["date"] = pd.to_datetime(train_data["date"])
        train_data = train_data.set_index("date")
        train_data = train_data.asfreq("W-MON")

        forecast_data["date"] = pd.to_datetime(forecast_data["date"])
        forecast_data = forecast_data.set_index("date")
        forecast_data = forecast_data.asfreq("W-MON")

        # Extract the unit_sales series
        train_series = train_data["unit_sales"].copy()
        forecast_series = forecast_data["unit_sales"].copy()

        # Optimize parameters
        best_alpha, best_beta, best_gamma = None, None, None
        best_trend, best_damped, best_seasonal = None, None, None
        best_mape = float("inf")
        no_improve_count = 0

        for comb in abg_combinations:
            trend, damped, seasonal, alpha, beta, gamma = comb

            try:
                model = ExponentialSmoothing(
                    train_series,
                    trend=trend,
                    damped_trend=damped,
                    seasonal=seasonal,
                    seasonal_periods=seasonal_periods,
                )

                fitted_model = model.fit(
                    smoothing_level=alpha,
                    smoothing_slope=beta,
                    smoothing_seasonal=gamma,
                )

                # Forecast the length of the forecast data
                forecast_length = len(forecast_series)

                y_pred = fitted_model.forecast(forecast_length)

                mape = mean_absolute_percentage_error(
                    forecast_series, y_pred[:forecast_length]
                )

                if mape < best_mape - early_stopping_threshold:

                    best_alpha, best_beta, best_gamma = alpha, beta, gamma
                    best_trend, best_damped, best_seasonal = trend, damped, seasonal
                    best_mape = mape

                    no_improve_count = 0  # Reset counter if improvement found
                else:
                    no_improve_count += 1

                # Early stopping if no improvement for max_no_improve_iters iterations
                if no_improve_count >= max_no_improve_iters:

                    print(
                        f"Early stopping for store_nbr {store_nbr}, item_nbr {item_nbr} due to no improvement."
                    )
                    break

            except Exception as e:
                print(
                    f"Failed to fit model for store_nbr {store_nbr}, item_nbr {item_nbr} with parameters {comb}: {e}"
                )
                continue

        # Fit the model with the best parameters
        forecast = None
        fitted_values = None

        try:
            model = ExponentialSmoothing(
                train_series,
                trend=best_trend,
                damped_trend=best_damped,
                seasonal=best_seasonal,
                seasonal_periods=seasonal_periods,
            )

            fitted_model = model.fit(
                smoothing_level=best_alpha,
                smoothing_slope=best_beta,
                smoothing_seasonal=best_gamma,
            )

            forecast = fitted_model.forecast(len(forecast_series))
            fitted_values = fitted_model.fittedvalues

        except Exception as e:
            print(
                f"Model failed for store_nbr {store_nbr}, item_nbr {item_nbr} with best parameters: {e}"
            )

        return (
            store_nbr,
            item_nbr,
            forecast,
            fitted_values,
            best_trend,
            best_damped,
            best_seasonal,
            best_alpha,
            best_beta,
            best_gamma,
            best_mape,
        )

    except Exception as e:
        print(f"Exception occurred for store_nbr {store_nbr}, item_nbr {item_nbr}: {e}")
        return (
            store_nbr,
            item_nbr,
            None,
            None,
            None,
            None,
            None,
            None,
            None,
            None,
            None,
        )

In [ ]:
import traceback

In [ ]:
def holt_winters_model_grid_forecast(
    train_df,
    forecast_df,
    seasonal_periods=52,
    early_stopping_threshold=0.01,
    max_no_improve_iters=50,
):

    # Unique store and item combinations
    unique_stores = train_df["store_nbr"].unique()
    unique_items = train_df["item_nbr"].unique()

    # Create a param grid for trend, dampened, seasonality, alpha, beta, and gamma
    t_params = ["add", "mul", None]
    d_params = [True, False]
    s_params = ["add", "mul", None]
    alphas = betas = gammas = np.arange(0, 1.00, 0.1)

    abg_combinations = list(
        itertools.product(t_params, d_params, s_params, alphas, betas, gammas)
    )

    forecasts = {}
    fitted_values = {}
    best_params = {}

    # Use multiprocessing
    with ProcessPoolExecutor() as executor:

        future_to_store_item = {
            executor.submit(
                process_item,
                store_nbr,
                item_nbr,
                train_df,
                forecast_df,
                abg_combinations,
                seasonal_periods,
                max_no_improve_iters,
                early_stopping_threshold,
            ): (store_nbr, item_nbr)
            for store_nbr in unique_stores
            for item_nbr in unique_items
        }

        for future in as_completed(future_to_store_item):

            store_nbr, item_nbr = future_to_store_item[future]

            try:
                (
                    store_nbr,
                    item_nbr,
                    forecast,
                    fitted_vals,
                    best_trend,
                    best_damped,
                    best_seasonal,
                    best_alpha,
                    best_beta,
                    best_gamma,
                    best_mape,
                ) = future.result()

                if forecast is not None:

                    forecasts[(store_nbr, item_nbr)] = forecast
                    fitted_values[(store_nbr, item_nbr)] = fitted_vals

                    best_params[(store_nbr, item_nbr)] = {
                        "trend": best_trend,
                        "damped": best_damped,
                        "seasonal": best_seasonal,
                        "alpha": best_alpha,
                        "beta": best_beta,
                        "gamma": best_gamma,
                        "mape": best_mape,
                    }

            except Exception as e:
                print(
                    f"Exception occurred for store_nbr {store_nbr}, item_nbr {item_nbr}:{str(e)}"
                )
                traceback.print_exc()

    return forecasts, fitted_values, best_params

In [ ]:
forecasts, fitted_values, best_params = holt_winters_model_grid_forecast(
    train_df_store_10_103520, val_df_store_10_103520
)

In [ ]:
# Evaluate forecasts and print metrics
metrics_df, average_store_metrics_df, overall_metrics_df = evaluate_forecasts(
    forecasts, val_df
)

Viz

In [ ]:
def advanced_forecast_visualization(series=df, forecast, actual):
    plt.figure(figsize=(15,6))
    plt.plot(series.index, series, label='Historical')
    plt.plot(forecast.index, forecast, label='Forecast', color='red')
    plt.fill_between(forecast.index, 
                     forecast - 1.96 * forecast.std(), 
                     forecast + 1.96 * forecast.std(), 
                     alpha=0.3, color='red')
    plt.title('Forecast with 95% Confidence Interval')
    plt.legend()